# OpenStreetMap Amenity Data Analysis
* Desc: Acquire and visualize number of amenities per European city using OpenStreetMap
* Data source: Overpass API (OpenStreetMap data)
* Goal: Count amenities in major European cities and create visualizations similar to cost_of_living analysis

## Setup and Imports

In [ ]:
import os
os.environ['PROJ_LIB'] = '${HOME}/miniconda3/share/proj'

import polars as pl
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap
from config import (
    Continent, european_city_coords, famous_european_cities,
    european_city_populations
)
import time
import requests
import json

# Create output directory
os.makedirs('data/output', exist_ok=True)

print('Setup complete')

## Query Overpass API for Cinema Data

In [ ]:
# Query OpenStreetMap Overpass API for cinemas - BY CITY NAME
OVERPASS_URL = 'https://overpass-api.de/api/interpreter'
AMENITY = 'library'
AMENITY_CAP = AMENITY.capitalize()
CACHE_FILE = f'data/output/{AMENITY}_counts.json'

cities_to_analyze = famous_european_cities
# cities_to_analyze = ['Berlin', 'London', 'Paris']
amenity_data = []

# Check if cached data exists
if os.path.exists(CACHE_FILE):
    print(f'Loading cached data from {CACHE_FILE}')
    with open(CACHE_FILE, 'r') as f:
        amenity_data = json.load(f)
    print(f'Loaded {len(amenity_data)} cities from cache\n')
else:
    print(f'Querying {len(cities_to_analyze)} cities from Overpass API...\n')
    print(f'Amenity type: {AMENITY}\n')
    
    for idx, city in enumerate(cities_to_analyze):
        print(f'[{idx + 1}/{len(cities_to_analyze)}] {city}: Querying...', end=' ')
        try:
            # Get city coordinates for reference
            matching_coords = None
            for key, coords in european_city_coords.items():
                if key.startswith(city + ','):
                    matching_coords = coords
                    break
            
            if matching_coords is None:
                continue
            
            lat, lon = matching_coords
            
            # Query by city name using area selector
            # area["name"="CityName"] finds the administrative area with that name
            query = f'''[out:json];
area["name"="{city}"]->.searchArea;
(
  node["amenity"="{AMENITY}"](area.searchArea);
  way["amenity"="{AMENITY}"](area.searchArea);
  relation["amenity"="{AMENITY}"](area.searchArea);
);
out body;'''
            
            # Execute query
            response = requests.post(
                OVERPASS_URL,
                data={'data': query},
                timeout=60,
                headers={'User-Agent': f'Mapy{AMENITY_CAP}Analysis/1.0'}
            )
            
            if response.status_code == 200:
                try:
                    result = response.json()
                    elements = result.get('elements', [])
                    amenity_count = len(elements)
                    
                    print(f'\tFound {amenity_count} {AMENITY_CAP}(s)')

                    amenity_data.append({
                        'City': city,
                        'Latitude': lat,
                        'Longitude': lon,
                        f'{AMENITY_CAP}_Count': amenity_count
                    })
                except json.JSONDecodeError:
                    pass
            else:
                print(f'\tError: HTTP {response.status_code}')
            
            # Respect API rate limits
            time.sleep(1)
            
        except Exception as e:
            pass
    
    if not amenity_data:
        raise RuntimeError(f'Failed to collect any data. Queried {len(cities_to_analyze)} cities but all failed.')
    
    print(f'\n✓ Successfully queried {len(amenity_data)} cities')
    
    # Save results
    with open(CACHE_FILE, 'w') as f:
        json.dump(amenity_data, f, indent=2)
    print(f'✓ Saved to {CACHE_FILE}')

## Process and Analyze Amenity Data

In [ ]:
# Process and analyze amenity data with population
if not amenity_data:
    raise RuntimeError('No amenity data available. Run the query cell first.')

# Convert to Polars DataFrame
df_amenities = pl.DataFrame(amenity_data)

# Add population column
df_amenities = df_amenities.with_columns([
    pl.col('City').map_elements(
        lambda city: european_city_populations.get(city, None),
        return_dtype=pl.Int64
    ).alias('Population')
])

# Drop cities with zero amenities or missing population
df_amenities = df_amenities.filter(
    (pl.col(f'{AMENITY_CAP}_Count') > 0) & (pl.col('Population').is_not_null())
)

# Calculate per-capita metrics
df_amenities = df_amenities.with_columns([
    (pl.col(f'{AMENITY_CAP}_Count') / pl.col('Population') * 10000).alias(f'{AMENITY_CAP}_Per_10k'),
    (pl.col(f'{AMENITY_CAP}_Count') / pl.col('Population') * 100000).alias(f'{AMENITY_CAP}_Per_100k'),
])

# Calculate Amenity Index (normalized 0-100)
df_amenities = df_amenities.with_columns([
    (
        (pl.col(f'{AMENITY_CAP}_Count') - pl.col(f'{AMENITY_CAP}_Count').min()) /
        (pl.col(f'{AMENITY_CAP}_Count').max() - pl.col(f'{AMENITY_CAP}_Count').min()) * 100
    ).alias(f'{AMENITY_CAP}_Index')
])

print(f'Dataset: {df_amenities.shape[0]} cities')
print(f'\n{AMENITY_CAP} Count Statistics:')
print(df_amenities.select([f'{AMENITY_CAP}_Count']).describe())

print(f'\n{AMENITY_CAP} Per 10,000 Inhabitants Statistics:')
print(df_amenities.select([f'{AMENITY_CAP}_Per_10k']).describe())

print(f'\nTop 15 Cities by {AMENITY_CAP} Count:')
print(df_amenities.sort(f'{AMENITY_CAP}_Count', descending=True).head(15).select(
    ['City', f'{AMENITY_CAP}_Count', 'Population', f'{AMENITY_CAP}_Per_10k']
))

print(f'\nTop 15 Cities by {AMENITY_CAP} Per 10,000 Inhabitants:')
print(df_amenities.sort(f'{AMENITY_CAP}_Per_10k', descending=True).head(15).select(
    ['City', f'{AMENITY_CAP}_Count', 'Population', f'{AMENITY_CAP}_Per_10k']
))

## Visualizations

In [ ]:
# Library Distribution Map (Per 10,000 Inhabitants)
SHAPE_FILE = './data/natural_earth/ne_10m_admin_0_countries'
FIGSIZE = (20, 14)

# City label shifts for clarity
CITIES_TO_SHIFT = {
    'Bratislava': (+15000, -15000, ''),
    'Stockholm': (+15000, -15000, ''),
    'Vienna': (-15000, +15000, 'top'),
    'Prague': (0, -10000, 'top'),
    'Amsterdam': (0, -20000, 'top'),
    'Paris': (0, -15000, 'top'),
}

region = Continent.europe
fig = plt.figure(figsize=FIGSIZE)
ax = fig.add_subplot(111)

m = Basemap(
    llcrnrlon=region['llcrnrlon'],
    llcrnrlat=region['llcrnrlat'] - 1.0,
    urcrnrlon=region['urcrnrlon'] - 12.0,
    urcrnrlat=region['urcrnrlat'],
    lat_0=region['lat_0'],
    lon_0=region['lon_0'],
    lat_1=region['lat_1'],
    lat_2=region['lat_2'],
    projection='lcc',
    resolution='i',
    area_thresh=1000,
)

m.drawmapboundary(fill_color='#e6f2ff', linewidth=0.5)
m.fillcontinents(color='#f0e6d2', lake_color='#e6f2ff')
m.drawcountries(linewidth=1.5, color='#666666')
m.drawcoastlines(linewidth=0.5, color='#999999')
m.readshapefile(SHAPE_FILE, 'countries', linewidth=1, color='#999999')

df_sorted = df_amenities.sort(f'{AMENITY_CAP}_Per_10k', descending=True)
x, y = m(df_sorted['Longitude'].to_list(), df_sorted['Latitude'].to_list())

# Scale sizes by per-capita metric
per_10k_values = df_sorted[f'{AMENITY_CAP}_Per_10k'].to_list()
sizes = [max(val ** 1.5 * 1000, 30) for val in per_10k_values]

scatter = m.scatter(
    x, y, s=sizes, alpha=0.75, c=per_10k_values,
    cmap='RdYlGn', edgecolors='#333333', linewidth=2, zorder=5, vmin=0
)

for xi, yi, city, per_10k in zip(x, y, df_sorted['City'].to_list(), per_10k_values):
    # Value in center of circle
    plt.text(xi, yi, f'{per_10k:.1f}', ha='center', va='center', 
             fontsize=8, fontweight='bold', color='#000000', zorder=6)
    
    # City label
    shift = CITIES_TO_SHIFT.get(city, (0, 0, ''))
    xi_shift = xi + shift[0]
    yi_shift = yi - 30000 + shift[1]
    plt.text(xi_shift, yi_shift, city, ha='center', va='top', fontsize=7, zorder=6,
             fontweight='500', color='#1a1a1a',
             bbox=dict(facecolor='#ffffff', alpha=0.85, edgecolor='#333333', 
                      boxstyle='round,pad=0.4', linewidth=1))

cbar = plt.colorbar(scatter, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label(f'{AMENITY_CAP} per 10,000 Inhabitants', fontsize=13, fontweight='bold')
cbar.ax.tick_params(labelsize=11)

plt.title(
    f'{AMENITY_CAP} Availability in European Cities (2026)\nPer 10,000 Inhabitants | Data: OpenStreetMap',
    fontsize=18, fontweight='bold', pad=25, color='#1a1a1a'
)

plt.tight_layout()
plt.savefig(f'data/output/europe_{AMENITY}s_per_capita_map.png', dpi=300, bbox_inches='tight', facecolor='white')
print(f'Saved: data/output/europe_{AMENITY}s_per_capita_map.png')
plt.show()

## Summary

In [ ]:
print(f'''
=== AMENITY DATA ANALYSIS SUMMARY ===

Total cities analyzed: {len(df_amenities)}
Total {AMENITY}s found: {int(df_amenities[f'{AMENITY_CAP}_Count'].sum())}
Average {AMENITY}s per city: {df_amenities[f'{AMENITY_CAP}_Count'].mean():.1f}
Median {AMENITY}s per city: {df_amenities[f'{AMENITY_CAP}_Count'].median():.1f}
Max {AMENITY}s in single city: {int(df_amenities[f'{AMENITY_CAP}_Count'].max())}
Min {AMENITY}s in single city: {int(df_amenities[f'{AMENITY_CAP}_Count'].min())}

=== PER CAPITA METRICS ===
Average {AMENITY}s per 10,000 inhabitants: {df_amenities[f'{AMENITY_CAP}_Per_10k'].mean():.2f}
Median {AMENITY}s per 10,000 inhabitants: {df_amenities[f'{AMENITY_CAP}_Per_10k'].median():.2f}
Max {AMENITY}s per 10,000 inhabitants: {df_amenities[f'{AMENITY_CAP}_Per_10k'].max():.2f}
Min {AMENITY}s per 10,000 inhabitants: {df_amenities[f'{AMENITY_CAP}_Per_10k'].min():.2f}

=== DATA SOURCE ===
Source: OpenStreetMap (Overpass API)
Amenities: node[amenity="{AMENITY}"], way[amenity="{AMENITY}"]
Population data: Wikipedia, national statistical offices, city government data (2024/2025)

=== OUTPUT FILES ===
- data/output/{AMENITY}_counts.json (raw data)
- data/output/europe_{AMENITY}s_map.png (map visualization)
- data/output/europe_{AMENITY}s_top_cities.png (bar chart)
''')